# HANDLING EMOJIS AND ABBREVATIONS USING LSTM


In [ ]:
# -*- coding: utf-8 -*-
"""
Final Advanced Sarcasm and Humor Detector
This script trains a sophisticated NLP model using an Embedding layer and an LSTM network
to classify text as sarcastic, humorous, or neutral.
"""

# Step 0: Install necessary libraries
!pip install emoji --quiet

# Step 1: Import all required libraries
import pandas as pd
import numpy as np
import re
import emoji
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("TensorFlow Version:", tf.__version__)

# Step 2: Load the original datasets
try:
    # Make sure to upload these files to your Colab session's file directory
    df_sarcasm = pd.read_csv('/content/drive/MyDrive/NLP prj/sarcasm/train-balanced-sarcasm.csv')
    df_jokes = pd.read_csv('/content/drive/MyDrive/NLP prj/humor/shortjokes.csv')
    print("✅ Datasets loaded successfully!")
except FileNotFoundError:
    print("❌ Error: Make sure 'train-balanced-sarcasm.csv' and 'shortjokes.csv' are uploaded to your Colab session!")
    # Dummy dataframes to prevent script from crashing if files are not found
    df_sarcasm = pd.DataFrame(columns=['comment', 'label'])
    df_jokes = pd.DataFrame(columns=['Joke'])


# Step 3: Define the Advanced Preprocessing Function
# Expanded dictionary for better handling of social media slang
slang_dict = {
    "lol": "laughing out loud",
    "lmfao": "laughing my freaking ass off",
    "smh": "shaking my head",
    "imo": "in my opinion",
    "ikr": "I know right",
    "omg": "oh my god",
    "btw": "by the way",
    "idk": "I don't know",
    "tbh": "to be honest",
    "ftw": "for the win",
    "irl": "in real life"
}

def preprocess_text(text):
    """
    Cleans and prepares text data for NLP models.
    Handles emojis, slang, URLs, mentions, and other noise.
    """
    if not isinstance(text, str):
        return ""
    # Convert emojis to their text description
    text = emoji.demojize(text, delimiters=(" ", " "))
    # Convert to lowercase
    text = text.lower()
    # Replace slang
    words = text.split()
    expanded_words = [slang_dict.get(word, word) for word in words]
    text = " ".join(expanded_words)
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (but keep the text)
    text = re.sub(r'#', '', text)
    # Remove non-alphanumeric characters (optional, can help clean noise)
    text = re.sub(r'[^a-z0-9\s_]', '', text)
    # Remove extra whitespace
    text = " ".join(text.split())
    return text

print("✅ Preprocessing function defined with expanded slang dictionary.")


# Step 4: Prepare and Preprocess the DataFrames
# Drop null values
df_sarcasm.dropna(subset=['comment'], inplace=True)
df_jokes.dropna(subset=['Joke'], inplace=True)

# Apply the preprocessing function
print("\nPreprocessing text data... (This may take a minute)")
df_sarcasm['comment'] = df_sarcasm['comment'].apply(preprocess_text)
df_jokes['Joke'] = df_jokes['Joke'].apply(preprocess_text)
print("✅ Text preprocessing complete.")

# Prepare sarcasm data (label 0 = neutral, label 1 = sarcastic)
df_sarcasm_processed = df_sarcasm[['comment', 'label']].copy()

# Prepare humor data (label 2 = humor)
df_jokes_processed = df_jokes[['Joke']].copy()
df_jokes_processed.rename(columns={'Joke': 'comment'}, inplace=True)
df_jokes_processed['label'] = 2


# Step 5: Combine Data and Convert to Numerical Sequences
# Combine all data into a single DataFrame
combined_df = pd.concat([df_sarcasm_processed, df_jokes_processed], ignore_index=True)

# Separate comments and labels
comments = combined_df['comment']
labels = combined_df['label']

# Initialize and fit the Keras Tokenizer
max_words = 15000  # Size of our vocabulary
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(comments)

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(comments)

# Pad sequences to ensure they are all the same length
maxlen = 100  # Max number of words per comment
X = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
y = labels.values

print("\nShape of data tensor (X):", X.shape)
print("Shape of label tensor (y):", y.shape)


# Step 6: Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("\n✅ Data split into training and testing sets.")
print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])


# Step 7: Build the Advanced LSTM Model
print("\nBuilding the LSTM model...")
model = Sequential([
    # Embedding layer: Turns word indices into dense vectors of a fixed size.
    Embedding(input_dim=max_words, output_dim=128, input_length=maxlen),
    # SpatialDropout1D: Regularization to prevent overfitting.
    SpatialDropout1D(0.3),
    # LSTM layer: Processes sequences, capturing contextual information.
    LSTM(64, dropout=0.3, recurrent_dropout=0.3),
    # Dense output layer: For classification.
    Dense(3, activation='softmax')  # 3 classes for neutral, sarcastic, humor
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.5 MB/s eta 0:00:00
TensorFlow Version: 2.19.0
✅ Datasets loaded successfully!
✅ Preprocessing function defined with expanded slang dictionary.

Preprocessing text data... (This may take a minute)
✅ Text preprocessing complete.

Shape of data tensor (X): (1242428, 100)
Shape of label tensor (y): (1242428,)

✅ Data split into training and testing sets.
Training set size: 993942
Testing set size: 248486

Building the LSTM model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting model training...
Epoch 1/2
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1053s 336ms/step - accuracy: 0.4053 - loss: 1.0469 - val_accuracy: 0.4066 - val_loss: 1.0445
Epoch 2/2
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1038s 334ms/step - accuracy: 0.4057 - loss: 1.0452 - val_accuracy: 0.4078 - val_loss: 1.0444
✅ Model training complete.

Evaluating the model on the test set...
7766/7766 ━━━━━━━━━━━━━━━━━━━━ 473s 61ms/step
Accuracy: 0.4068
Precision: 0.1655
Recall: 0.4068
F1-score: 0.2353

--- Testing with new comments ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Comment: 'Oh great, another meeting that could have been an email 🙄'
-> Predicted: Not Sarcastic/Humorous

Comment: 'Why did the scarecrow win an award? Because he was outstanding in his field!'
-> Predicted: Not Sarcastic/Humorous

Comment: 'I'm so excited for the weekend!'
-> Predicted: Not Sarcastic/Humorous

Comment: 'lmfao I just love it when my code breaks for no reason smh'
-> Predicted: Not Sarcastic/Humorous



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:

# Step 8: Train the Model
print("\nStarting model training...")
history = model.fit(X_train, y_train,
                    epochs=5,  # Increase epochs for better performance, but 2 is good for a quick run
                    batch_size=256,
                    validation_split=0.2,
                    verbose=1)
print("✅ Model training complete.")


# Step 9: Evaluate the Model on the Test Set
print("\nEvaluating the model on the test set...")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")


# Step 10: Make Predictions on New, Unseen Data
print("\n--- Testing with new comments ---")
new_comments = [
    "Oh great, another meeting that could have been an email 🙄", # Sarcastic
    "Why did the scarecrow win an award? Because he was outstanding in his field!", # Humor
    "I'm so excited for the weekend!", # Neutral
    "lmfao I just love it when my code breaks for no reason smh" # Sarcastic
]

# The prediction pipeline
def predict_text(text_list):
    # Preprocess the text
    processed_texts = [preprocess_text(t) for t in text_list]
    # Convert to sequences
    sequences = tokenizer.texts_to_sequences(processed_texts)
    # Pad the sequences
    padded_sequences = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')
    # Make predictions
    predictions = model.predict(padded_sequences)
    # Get the class with the highest probability
    predicted_classes = np.argmax(predictions, axis=1)
    return predicted_classes

# Run predictions
predictions = predict_text(new_comments)

# Display results
label_map = {0: "Not Sarcastic/Humorous", 1: "Sarcastic", 2: "Humor"}
for i, comment in enumerate(new_comments):
    print(f"Comment: '{comment}'")
    print(f"-> Predicted: {label_map[predictions[i]]}\n")


Starting model training...
Epoch 1/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1053s 339ms/step - accuracy: 0.4058 - loss: 1.0456 - val_accuracy: 0.4078 - val_loss: 1.0443
Epoch 2/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1036s 334ms/step - accuracy: 0.4070 - loss: 1.0449 - val_accuracy: 0.4078 - val_loss: 1.0443
Epoch 3/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1002s 323ms/step - accuracy: 0.4073 - loss: 1.0448 - val_accuracy: 0.4078 - val_loss: 1.0444
Epoch 4/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1052s 326ms/step - accuracy: 0.5127 - loss: 0.9153 - val_accuracy: 0.7036 - val_loss: 0.6301
Epoch 5/5
3107/3107 ━━━━━━━━━━━━━━━━━━━━ 1017s 327ms/step - accuracy: 0.7079 - loss: 0.6197 - val_accuracy: 0.7224 - val_loss: 0.5878
✅ Model training complete.

Evaluating the model on the test set...
7766/7766 ━━━━━━━━━━━━━━━━━━━━ 453s 58ms/step
Accuracy: 0.7243
Precision: 0.7246
Recall: 0.7243
F1-score: 0.7238

--- Testing with new comments ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Comment: 'Oh great, another meeting that could h

# saving the model


In [ ]:
import pickle
import os

# Define the directory to save your files
save_dir = 'model_artifacts'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 1. Save the trained Keras model
# We use the recommended '.keras' format
model_path = os.path.join(save_dir, 'sarcasm_humor_model.keras')
model.save(model_path)
print(f"✅ Model saved successfully to: {model_path}")

# 2. Save the fitted Tokenizer
# We use pickle to save the tokenizer object
tokenizer_path = os.path.join(save_dir, 'tokenizer.pkl')
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"✅ Tokenizer saved successfully to: {tokenizer_path}")

✅ Model saved successfully to: model_artifacts/sarcasm_humor_model.keras
✅ Tokenizer saved successfully to: model_artifacts/tokenizer.pkl
